In [3]:
import pandas as pd

# 1. Load the dataset
df_types = pd.read_csv('deaths-in-armed-conflicts-by-type.csv')

# 2. Filter for global data ('World' entity) and years from 1946 onwards
df_world = df_types[(df_types['Entity'] == 'World') & (df_types['Year'] >= 1946)].copy()

# 3. Select only the necessary columns (Year and the conflict types)
columns_to_keep = [
    'Year',
    'Interstate',           # war between countries
    'Intrastate',           # internal war
    'One-sided violence',   # violence
    'Non-state'             # non-state conflicts
]
df_streamgraph = df_world[columns_to_keep]

# 4. Sort by Year
df_streamgraph = df_streamgraph.sort_values(by='Year')

# 5. Save the clean data for D3.js
df_streamgraph.to_csv('chart1_streamgraph.csv', index=False)

print("Chart 1 Data Saved! Here are the first 3 rows:")
print(df_streamgraph.head(3))

Chart 1 Data Saved! Here are the first 3 rows:
      Year  Interstate  Intrastate  One-sided violence  Non-state
7416  1989         868       54316                7992       4170
7417  1990        1086       79211                9863       5251
7418  1991       21889       48464                9739       3860


In [4]:
import pandas as pd

# 1. Load the dataset
df_onesided = pd.read_csv('deaths-from-one-sided-violence.csv')

# 2. Filter for recent decade (2012 onwards)
df_recent = df_onesided[df_onesided['Year'] >= 2012].copy()
df_recent = df_recent.rename(columns={'Best estimate': 'Fatalities'})

# 3. Group by Country and Sum
df_map = df_recent.groupby(['Entity', 'Code'])['Fatalities'].sum().reset_index()

# Remove 'World', Continents, and invalid codes
df_map = df_map.dropna(subset=['Code'])
df_map = df_map[df_map['Code'].str.len() == 3]

# 5. Filter out zeros and Sort
df_map = df_map[df_map['Fatalities'] > 0]
df_map = df_map.sort_values(by='Fatalities', ascending=False)

# 6. Save to CSV
output_file = 'chart2_choropleth.csv'
df_map.to_csv(output_file, index=False)

print("\nSample of prepared data (Top 5 worst COUNTRIES only):")
print(df_map.head())
print(f"\nData saved to: {output_file}")


Sample of prepared data (Top 5 worst COUNTRIES only):
                           Entity Code  Fatalities
47   Democratic Republic of Congo  COD       19245
130                       Nigeria  NGA       14369
175                         Syria  SYR       13838
61                       Ethiopia  ETH       12132
84                           Iraq  IRQ       11423

Data saved to: chart2_choropleth.csv


In [5]:
import pandas as pd

#extended_choropleth_per_capita

df_conflict = pd.read_csv('chart2_choropleth.csv')
df_pop = pd.read_csv('population-unwpp.csv')

print("Files loaded successfully.")


latest_year = df_pop['Year'].max()
print(f"Using population data from year: {latest_year}")


df_pop_recent = df_pop[df_pop['Year'] == latest_year].copy()


pop_col = [col for col in df_pop_recent.columns if 'Population' in col][0]
df_pop_recent = df_pop_recent[['Code', pop_col]]
df_pop_recent.columns = ['Code', 'Population']


df_merged = pd.merge(df_conflict, df_pop_recent, on='Code', how='left')


df_merged['Population'] = df_merged['Population'].fillna(0)


df_merged['Fatalities_Per_100k'] = df_merged.apply(
    lambda row: (row['Fatalities'] / row['Population']) * 100000 if row['Population'] > 0 else 0,
    axis=1
)


df_merged = df_merged.sort_values(by='Fatalities_Per_100k', ascending=False)


output_filename = 'chart2_choropleth_extended.csv'
df_merged.to_csv(output_filename, index=False)

print(f"\n new file is created: {output_filename}")
print(df_merged[['Entity', 'Fatalities', 'Population', 'Fatalities_Per_100k']].head(10))

Files loaded successfully.
Using population data from year: 2023

 new file is created: chart2_choropleth_extended.csv
                          Entity  Fatalities  Population  Fatalities_Per_100k
5       Central African Republic        7420     5152415           144.010139
2                          Syria       13838    23594623            58.648956
11                   South Sudan        2915    11483370            25.384534
6                   Burkina Faso        5836    23025777            25.345507
4                           Iraq       11423    45074055            25.342739
0   Democratic Republic of Congo       19245   105789733            18.191746
9                           Mali        4324    23769130            18.191663
16                         Haiti        1651    11637402            14.187015
7                          Sudan        5679    50042800            11.348286
10                   Afghanistan        3947    41454762             9.521222


In [6]:
import pandas as pd

print("=== Chart 3: Lollipop Chart Data Preprocessing ===")

# 1. Load the Excel dataset
file_name = 'number_of_events_targeting_civilians_by_country_year_as_of_30Jan2026.xlsx'
df_events = pd.read_excel(file_name)

# 2. Filter for recent and complete years (2017 to 2025)
df_recent = df_events[(df_events['YEAR'] >= 2017) & (df_events['YEAR'] <= 2025)].copy()

# 3. Group by Country and Sum the 'EVENTS'
df_lollipop = df_recent.groupby('COUNTRY')['EVENTS'].sum().reset_index()

# 4. Sort in descending order to find the worst offenders
df_lollipop = df_lollipop.sort_values(by='EVENTS', ascending=False)

# 5. Keep only the Top 10 countries
df_top10 = df_lollipop.head(10)

# 6. Save the clean data as CSV
output_file = 'chart3_lollipop.csv'
df_top10.to_csv(output_file, index=False)

print("\n--- Top 10 Countries Prepared for Lollipop Chart ---")
print(df_top10)
print(f"\nData saved to: {output_file}")

=== Chart 3: Lollipop Chart Data Preprocessing ===

--- Top 10 Countries Prepared for Lollipop Chart ---
                          COUNTRY  EVENTS
138                        Mexico   47755
31                         Brazil   33442
213                         Syria   33034
166                     Palestine   26106
147                       Myanmar   18528
228                       Ukraine   17899
156                       Nigeria   15176
100                         India   14660
59   Democratic Republic of Congo   11751
50                       Colombia   11526

Data saved to: chart3_lollipop.csv
